# Audit Outcome

This notebook evaluates the manual audit and estimates the noise parameters the benchmark releases. The per-stratum posteriors derived here enter the resolution characterization as the perturbation applied to the released labels.

In [40]:
from benchmark_notebook.io import CloudStorage
from benchmark_notebook import theme

import polars as pl

store = CloudStorage(bucket="bug-request-bucket")

annotation = pl.from_arrow(
    store.asset(
        key="audit-change-annotation",
        version="1.0.0",
        location="main",
        partition="default",
    )
)

# The failure label is derived from the raw annotation fields, not the stored
# verdict column: a change fails stage 1 with no intrinsic correction, stage 2 with
# a wrong introducer, and otherwise passes to the node stage.
annotation = annotation.with_columns(
    pl.when(~pl.col("categories").list.contains("intrinsic"))
    .then(pl.lit("stage_1"))
    .when(pl.col("attribution") == "wrong")
    .then(pl.lit("stage_2"))
    .otherwise(pl.lit("passed"))
    .alias("stage")
)

realized = annotation["rho_bin"].value_counts().sort("rho_bin")
print(
    f"annotation={annotation.height} rows over {annotation['repository'].n_unique()} repositories"
)
print(annotation.select("repository", "number", "rho_bin", "stage").head(8))
print(realized)

annotation=100 rows over 6 repositories
shape: (8, 4)
┌─────────────────┬────────┬─────────┬─────────┐
│ repository      ┆ number ┆ rho_bin ┆ stage   │
│ ---             ┆ ---    ┆ ---     ┆ ---     │
│ str             ┆ i64    ┆ i64     ┆ str     │
╞═════════════════╪════════╪═════════╪═════════╡
│ microsoft/Olive ┆ 2355   ┆ 0       ┆ passed  │
│ microsoft/Olive ┆ 2438   ┆ 0       ┆ passed  │
│ microsoft/Olive ┆ 1018   ┆ 0       ┆ passed  │
│ microsoft/Olive ┆ 916    ┆ 0       ┆ stage_1 │
│ microsoft/Olive ┆ 2313   ┆ 0       ┆ passed  │
│ microsoft/Olive ┆ 1975   ┆ 0       ┆ passed  │
│ microsoft/Olive ┆ 2317   ┆ 0       ┆ stage_2 │
│ microsoft/Olive ┆ 2229   ┆ 0       ┆ passed  │
└─────────────────┴────────┴─────────┴─────────┘
shape: (3, 2)
┌─────────┬───────┐
│ rho_bin ┆ count │
│ ---     ┆ ---   │
│ i64     ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 17    │
│ 1       ┆ 48    │
│ 2       ┆ 35    │
└─────────┴───────┘


## Released population

The audit was drawn before the released population was fully constrained, so two of the drawn changes fall outside it: their defect sits off the reconstructed integration branch, which the single-branch assumption of the noise model does not cover. The label asset carries that restriction along with the language scope, so the audited changes are cut to it by a semi-join and every estimate below is computed on the remaining sample. The excluded changes are printed rather than absorbed silently into the stratum totals.

In [41]:
from google.cloud import storage
from benchmark.attribution.history import VersionControl
from benchmark.definitions.partitions import Shard
from benchmark.localization.projection import NodeProjection

# The released population is the label asset. It applies both restrictions the audit
# inherits -- a defect off the reconstructed integration branch, and a marking naming a
# file outside the analysed language -- so the audited changes are cut to it by a
# semi-join rather than by reproducing either rule here. Two of the drawn changes fall
# outside it; they are printed rather than absorbed silently into the stratum totals.
released = pl.from_arrow(
    store.asset(
        key="localization-change-label",
        version="1.0.0",
        location="main",
        partition="default",
        columns=["defect"],
    )
)
outside = annotation.join(
    released.select(pl.col("defect").alias("introducer")), on="introducer", how="anti"
)
print(outside.select("repository", "number", "rho_bin", "stage"))

annotation = annotation.join(
    released.select(pl.col("defect").alias("introducer")), on="introducer", how="semi"
)
realized = annotation["rho_bin"].value_counts().sort("rho_bin")
print(
    f"annotation={annotation.height} rows over {annotation['repository'].n_unique()} repositories"
)
print(realized)

# One projection per repository over the scope-source-raw bare repo, reused by the node
# lift and the N(c) enumeration below.
stationary = pl.from_arrow(
    store.asset(
        key="scope-repository-stationary",
        location="main",
        version="1.0.0",
        partition="default",
        columns=["name"],
    )
)
names = sorted(stationary["name"].to_list())
client = storage.Client(project="bug-request")
projections = {}


def projection(name):
    if name not in projections:
        projections[name] = NodeProjection(
            VersionControl.create(
                client,
                bucket="bug-request-bucket",
                key="scope-source-raw",
                location="main",
                version="4.0.0",
                partition=Shard.to_space(names, name),
            )
        )
    return projections[name]

shape: (2, 4)
┌──────────────────────────┬────────┬─────────┬─────────┐
│ repository               ┆ number ┆ rho_bin ┆ stage   │
│ ---                      ┆ ---    ┆ ---     ┆ ---     │
│ str                      ┆ i64    ┆ i64     ┆ str     │
╞══════════════════════════╪════════╪═════════╪═════════╡
│ kedro-org/kedro-plugins  ┆ 1136   ┆ 1       ┆ stage_1 │
│ mistralai/mistral-common ┆ 103    ┆ 2       ┆ passed  │
└──────────────────────────┴────────┴─────────┴─────────┘
annotation=98 rows over 6 repositories
shape: (3, 2)
┌─────────┬───────┐
│ rho_bin ┆ count │
│ ---     ┆ ---   │
│ i64     ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 17    │
│ 1       ┆ 47    │
│ 2       ┆ 34    │
└─────────┴───────┘


## Audit population and strata

The population is the confidence pool restricted exactly as the production sample was drawn, with the off-trunk exclusion applied. Bin edges are fixed by the design at three equally spaced bins rather than estimated, so removing changes shifts the stratum sizes but never the stratum membership of a retained change.

In [42]:
import json

from benchmark.audit.sampling import HybridAuditSampling

# The audit population is the confidence pool restricted exactly as audit_change_sample
# does: rho not null, the pair emitted at least one location (marked), and one row per
# defective change via rho = max over its pairs.
reliability = pl.from_arrow(
    store.asset(
        key="confidence-change-reliability", version="5.0.0", partition="default"
    )
).filter(pl.col("rho").is_not_null())

# locations is a per-trace scalar repeated on every trace row, so one row per
# (name, correction, defect) carries the marking that decides membership.
trace = pl.from_arrow(
    store.asset(
        key="localization-change-trace",
        version="3.1.0",
        partition="*",
        columns=["name", "correction", "defect", "locations"],
    )
).unique(subset=["name", "correction", "defect"])

marked = trace.filter(
    pl.col("locations").map_elements(
        lambda value: bool(value) and bool(json.loads(value)), return_dtype=pl.Boolean
    )
).select("name", "correction", "defect")

population = (
    reliability.join(marked, on=["name", "correction", "defect"], how="semi")
    .sort("rho", descending=True)
    .unique(subset=["name", "defect"], keep="first")
    .sort("name", "defect")
)

# The population is cut to the released one by the same semi-join, so B_j describes
# the changes the benchmark can legitimately draw from.
population = population.join(released, on="defect", how="semi")

sampler = HybridAuditSampling(fraction=0.75, bins=3, seed=451600)
rho = population["rho"].to_numpy()
strata = sampler.strata(rho)
population = population.with_columns(pl.Series("rho_bin", strata).cast(pl.Int64))

bin_size = population["rho_bin"].value_counts().sort("rho_bin")
print(f"population={population.height} defective changes")
print(bin_size)

population=360 defective changes
shape: (3, 2)
┌─────────┬───────┐
│ rho_bin ┆ count │
│ ---     ┆ ---   │
│ i64     ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 43    │
│ 1       ┆ 187   │
│ 2       ┆ 130   │
└─────────┴───────┘


## Allocation

Stratum sizes of the cleaned population against the realized sample. The draw itself is calibrated by the audit design and not reproduced here; the shortfall against the fixed budget is the off-trunk exclusion, applied after the draw was already annotated.

In [43]:
edges = ["[0, 1/3)", "[1/3, 2/3)", "[2/3, 1]"]

# B_j: the stratum sizes of the released population. Realized n_j: the strata the
# annotated sample landed in, after the same restriction. The gap to the budget the
# production draw selected is exactly the changes it should not have been able to see.
realized = annotation["rho_bin"].value_counts().rename({"count": "realized"})

allocation = (
    bin_size.rename({"count": "B_j"})
    .join(realized, on="rho_bin", how="left")
    .with_columns(
        pl.col("rho_bin").replace_strict(dict(enumerate(edges))).alias("rho_range")
    )
    .fill_null(0)
    .select("rho_bin", "rho_range", "B_j", "realized")
    .sort("rho_bin")
)

print(allocation)
print(
    f"B_j sum={allocation['B_j'].sum()}  realized sum={allocation['realized'].sum()}"
)

shape: (3, 4)
┌─────────┬────────────┬─────┬──────────┐
│ rho_bin ┆ rho_range  ┆ B_j ┆ realized │
│ ---     ┆ ---        ┆ --- ┆ ---      │
│ i64     ┆ str        ┆ u32 ┆ u32      │
╞═════════╪════════════╪═════╪══════════╡
│ 0       ┆ [0, 1/3)   ┆ 43  ┆ 17       │
│ 1       ┆ [1/3, 2/3) ┆ 187 ┆ 47       │
│ 2       ┆ [2/3, 1]   ┆ 130 ┆ 34       │
└─────────┴────────────┴─────┴──────────┘
B_j sum=360  realized sum=98


## Distribution across repositories

The design fixes the marginal over strata but leaves the repository margin to the draw, so uneven representation is expected and reported rather than corrected.

In [44]:
# Realized sample crossed by repository (K) and stratum (J), with the row total.
crosstab = (
    annotation.pivot(
        on="rho_bin", index="repository", values="number", aggregate_function="len"
    )
    .fill_null(0)
    .sort("repository")
)
crosstab = crosstab.select(
    "repository",
    *[pl.col(str(j)).alias(f"j={j}") for j in sorted(annotation["rho_bin"].unique())],
).with_columns(pl.sum_horizontal(pl.exclude("repository")).alias("total"))

with pl.Config(tbl_rows=-1, fmt_str_lengths=60):
    print(crosstab)
print(f"repositories={crosstab.height}  total={crosstab['total'].sum()}")

shape: (6, 5)
┌────────────────────────────────────────────┬─────┬─────┬─────┬───────┐
│ repository                                 ┆ j=0 ┆ j=1 ┆ j=2 ┆ total │
│ ---                                        ┆ --- ┆ --- ┆ --- ┆ ---   │
│ str                                        ┆ u32 ┆ u32 ┆ u32 ┆ u32   │
╞════════════════════════════════════════════╪═════╪═════╪═════╪═══════╡
│ OpenJobDescription/openjd-model-for-python ┆ 0   ┆ 0   ┆ 1   ┆ 1     │
│ awslabs/graphstorm                         ┆ 5   ┆ 17  ┆ 13  ┆ 35    │
│ cohere-ai/cohere-compass-sdk               ┆ 2   ┆ 2   ┆ 1   ┆ 5     │
│ kedro-org/kedro-plugins                    ┆ 1   ┆ 5   ┆ 1   ┆ 7     │
│ microsoft/Olive                            ┆ 9   ┆ 23  ┆ 17  ┆ 49    │
│ mistralai/mistral-common                   ┆ 0   ┆ 0   ┆ 1   ┆ 1     │
└────────────────────────────────────────────┴─────┴─────┴─────┴───────┘
repositories=6  total=98


## Stage exits

The audit applies the three stages in order and stops at the first failure, assigning the failure label $V_c$ — named `stage_1`, `stage_2`, and `passed` here. The exit distribution shows which directional error source dominates before the node decisions.

In [45]:
# Stage exits in procedure order: stage 1 (no intrinsic correction), stage 2 (wrong
# introducer), passed (reached the node stage).
stages_order = ["stage_1", "stage_2", "passed"]


def stage_exits(frame, index):
    counts = (
        frame.pivot(on="stage", index=index, values="number", aggregate_function="len")
        .fill_null(0)
        .sort(index)
    )
    counts = counts.select(
        index,
        *[
            pl.col(s).alias(s) if s in counts.columns else pl.lit(0).alias(s)
            for s in stages_order
        ],
    ).with_columns(pl.sum_horizontal(stages_order).alias("total"))
    return counts.with_columns(
        (pl.col("passed") / pl.col("total")).round(3).alias("pass_rate")
    )


stages = stage_exits(annotation, "rho_bin")
print(stages)
print(
    f"passed overall = {(annotation['stage'] == 'passed').sum()} / {annotation.height} = {(annotation['stage'] == 'passed').mean():.3f}"
)

shape: (3, 6)
┌─────────┬─────────┬─────────┬────────┬───────┬───────────┐
│ rho_bin ┆ stage_1 ┆ stage_2 ┆ passed ┆ total ┆ pass_rate │
│ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---   ┆ ---       │
│ i64     ┆ u32     ┆ u32     ┆ u32    ┆ u32   ┆ f64       │
╞═════════╪═════════╪═════════╪════════╪═══════╪═══════════╡
│ 0       ┆ 2       ┆ 1       ┆ 14     ┆ 17    ┆ 0.824     │
│ 1       ┆ 5       ┆ 4       ┆ 38     ┆ 47    ┆ 0.809     │
│ 2       ┆ 4       ┆ 0       ┆ 30     ┆ 34    ┆ 0.882     │
└─────────┴─────────┴─────────┴────────┴───────┴───────────┘
passed overall = 82 / 98 = 0.837


## Stage exits by repository

Whether the change-level error concentrates in individual repositories rather than spreading across the cohort.

In [46]:
by_repository = stage_exits(annotation, "repository")

with pl.Config(tbl_rows=-1, fmt_str_lengths=60):
    print(by_repository)

shape: (6, 6)
┌────────────────────────────────────────────┬─────────┬─────────┬────────┬───────┬───────────┐
│ repository                                 ┆ stage_1 ┆ stage_2 ┆ passed ┆ total ┆ pass_rate │
│ ---                                        ┆ ---     ┆ ---     ┆ ---    ┆ ---   ┆ ---       │
│ str                                        ┆ u32     ┆ u32     ┆ u32    ┆ u32   ┆ f64       │
╞════════════════════════════════════════════╪═════════╪═════════╪════════╪═══════╪═══════════╡
│ OpenJobDescription/openjd-model-for-python ┆ 0       ┆ 0       ┆ 1      ┆ 1     ┆ 1.0       │
│ awslabs/graphstorm                         ┆ 3       ┆ 2       ┆ 30     ┆ 35    ┆ 0.857     │
│ cohere-ai/cohere-compass-sdk               ┆ 2       ┆ 0       ┆ 3      ┆ 5     ┆ 0.6       │
│ kedro-org/kedro-plugins                    ┆ 0       ┆ 0       ┆ 7      ┆ 7     ┆ 1.0       │
│ microsoft/Olive                            ┆ 6       ┆ 3       ┆ 40     ┆ 49    ┆ 0.816     │
│ mistralai/mistral-common

## Node level (stage 3)

Markings are recorded at line resolution and lifted to the containing syntax node, the resolution the audit is defined at; a node counts rejected once any of its markings was. This yields $\tilde Y_c$, which feeds $\hat\alpha$, and the added set $A_c$, which feeds $\hat\gamma^{+}$.

In [47]:
import json


class NodeAudit:
    # N(c) lives in the base tree, so each marked line is lifted to the syntax node
    # that carries it, through the projections built above.
    def _projection(self, repository):
        return projection(repository)

    def marks(self, frame):
        # One row per marking, the atomic unit of the audit: a marked line with its
        # origin and status, and the node it lifts to as columns. Node counts,
        # accept/reject splits and type distributions are all aggregation on top and
        # are left to the reader. An unmappable line keeps a null node.
        records = []
        for row in frame.iter_rows(named=True):
            projected = self._projection(row["repository"])
            for block in json.loads(row["selections"]):
                origin = "manual" if block["origin"] == "manual" else "suggested"
                status = "accepted" if origin == "manual" else block["status"]
                for line in block["lines"]:
                    node = projected.project(
                        row["introducer"], block["path"], line["line"], line["side"]
                    )
                    records.append(
                        {
                            "repository": row["repository"],
                            "number": row["number"],
                            "rho_bin": row["rho_bin"],
                            "path": block["path"],
                            "line": line["line"],
                            "side": line["side"],
                            "origin": origin,
                            "status": status,
                            "type": None if node is None else node.type,
                            "node_start": None if node is None else list(node.start),
                            "node_end": None if node is None else list(node.end),
                        }
                    )
        return pl.DataFrame(records)


# Failure label from the raw fields (the verdict column is derived and unused): a
# change reaches the node stage when it has an intrinsic correction and a correct
# introducer.
resolved = annotation.filter(pl.col("stage") == "passed")
marks = NodeAudit().marks(resolved)
print(
    f"changes={resolved.height}  markings={marks.height}  unmappable={marks['type'].is_null().sum()}"
)
with pl.Config(tbl_rows=12, fmt_str_lengths=40):
    print(marks.head(12))

changes=82  markings=295  unmappable=3
shape: (12, 11)
┌──────────────┬────────┬─────────┬──────────────┬───┬──────────┬────────┬────────────┬────────────┐
│ repository   ┆ number ┆ rho_bin ┆ path         ┆ … ┆ status   ┆ type   ┆ node_start ┆ node_end   │
│ ---          ┆ ---    ┆ ---     ┆ ---          ┆   ┆ ---      ┆ ---    ┆ ---        ┆ ---        │
│ str          ┆ i64    ┆ i64     ┆ str          ┆   ┆ str      ┆ str    ┆ list[i64]  ┆ list[i64]  │
╞══════════════╪════════╪═════════╪══════════════╪═══╪══════════╪════════╪════════════╪════════════╡
│ microsoft/Ol ┆ 2355   ┆ 0       ┆ olive/passes ┆ … ┆ accepted ┆ block  ┆ [73, 4]    ┆ [139, 15]  │
│ ive          ┆        ┆         ┆ /onnx/common ┆   ┆          ┆        ┆            ┆            │
│              ┆        ┆         ┆ .py          ┆   ┆          ┆        ┆            ┆            │
│ microsoft/Ol ┆ 2438   ┆ 0       ┆ olive/packag ┆ … ┆ accepted ┆ block  ┆ [26, 8]    ┆ [36, 91]   │
│ ive          ┆        ┆         ┆ 

In [48]:
# Aggregate the markings up to nodes. A node is (change, path, span, type); several
# markings can hit one node, so a suggested node counts rejected if any of its
# markings was rejected. Ỹ_c = suggested nodes, A_c = manual nodes not already
# suggested.
mapped = marks.filter(pl.col("type").is_not_null())
node_marks = (
    mapped.group_by(
        "repository", "number", "rho_bin", "path", "node_start", "node_end", "type"
    )
    .agg(
        pl.col("origin")
        .min()
        .alias("origin"),  # "manual" < "suggested": suggested wins
        (pl.col("status") == "rejected").any().alias("any_rejected"),
    )
    .with_columns(
        pl.when(pl.col("origin") == "manual")
        .then(pl.lit("added"))
        .when(pl.col("any_rejected"))
        .then(pl.lit("rejected"))
        .otherwise(pl.lit("accepted"))
        .alias("decision")
    )
)

# Per stratum: marked nodes Ỹ_c (accepted + rejected), added nodes A_c.
node_strata = (
    node_marks.group_by("rho_bin")
    .agg(
        (pl.col("decision") != "added").sum().alias("marked"),
        (pl.col("decision") == "accepted").sum().alias("accepted"),
        (pl.col("decision") == "rejected").sum().alias("rejected"),
        (pl.col("decision") == "added").sum().alias("added"),
    )
    .with_columns((pl.col("rejected") / pl.col("marked")).round(3).alias("fdr_raw"))
    .sort("rho_bin")
)
print(node_strata)
print(node_marks["decision"].value_counts().sort("decision"))
marked_total = (node_marks["decision"] != "added").sum()
print(f"raw FDR = {(node_marks['decision'] == 'rejected').sum() / marked_total:.3f}")

# Node type by decision: which kinds of node were rejected, accepted, added.
with pl.Config(tbl_rows=-1):
    print(node_marks.group_by("type", "decision").len().sort("len", descending=True))

shape: (3, 6)
┌─────────┬────────┬──────────┬──────────┬───────┬─────────┐
│ rho_bin ┆ marked ┆ accepted ┆ rejected ┆ added ┆ fdr_raw │
│ ---     ┆ ---    ┆ ---      ┆ ---      ┆ ---   ┆ ---     │
│ i64     ┆ u32    ┆ u32      ┆ u32      ┆ u32   ┆ f64     │
╞═════════╪════════╪══════════╪══════════╪═══════╪═════════╡
│ 0       ┆ 19     ┆ 14       ┆ 5        ┆ 2     ┆ 0.263   │
│ 1       ┆ 104    ┆ 59       ┆ 45       ┆ 2     ┆ 0.433   │
│ 2       ┆ 47     ┆ 38       ┆ 9        ┆ 3     ┆ 0.191   │
└─────────┴────────┴──────────┴──────────┴───────┴─────────┘
shape: (3, 2)
┌──────────┬───────┐
│ decision ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ accepted ┆ 111   │
│ added    ┆ 7     │
│ rejected ┆ 59    │
└──────────┴───────┘
raw FDR = 0.347
shape: (31, 3)
┌──────────────────────┬──────────┬─────┐
│ type                 ┆ decision ┆ len │
│ ---                  ┆ ---      ┆ --- │
│ str                  ┆ str      ┆ u32 │
╞══════════════════════╪══════════╪

In [49]:
# N(c): every modified node of the introducer's diff against its parent, lifted
# through the same projection the markings use and deduplicated on the base identity.
# A deletion carries the base path and side, an addition the after path and side, so
# renames resolve correctly. This is the candidate space the omission rate needs.
def change_nodes(repository, commit):
    projected = projection(repository)
    vc = projected._repository  # noqa: SLF001
    base = vc.parents(commit)[0]
    entries = set()
    for patch in vc.diff(base, commit, context=0):
        new_path, old_path = patch.delta.new_file.path, patch.delta.old_file.path
        for hunk in patch.hunks:
            for entry in hunk.lines:
                if entry.origin == "-":
                    node = projected.project(
                        commit, old_path, entry.old_lineno, "base"
                    )
                elif entry.origin == "+":
                    node = projected.project(
                        commit, new_path, entry.new_lineno, "after"
                    )
                else:
                    continue
                if node is not None:
                    entries.add((node.path, node.start, node.end, node.type))
    return entries


nc = {
    (row["repository"], row["number"]): change_nodes(
        row["repository"], row["introducer"]
    )
    for row in annotation.iter_rows(named=True)
}
n_frame = pl.DataFrame(
    [
        {"repository": repo, "number": number, "n_c": len(nodes)}
        for (repo, number), nodes in nc.items()
    ]
)
print(
    f"N(c) enumerated for {n_frame.height} changes; |N(c)| median={int(n_frame['n_c'].median())} max={n_frame['n_c'].max()}"
)

N(c) enumerated for 98 changes; |N(c)| median=98 max=3911


In [50]:
# Consistency: every audited node (suggested or added) must lie in the enumerated N(c)
# of its change, otherwise enumeration and lift diverge and the denominator is wrong.
divergent = 0
for (repo, number), group in node_marks.group_by("repository", "number"):
    universe = nc.get((repo, number), set())
    for row in group.iter_rows(named=True):
        key = (
            row["path"],
            tuple(row["node_start"]),
            tuple(row["node_end"]),
            row["type"],
        )
        if key not in universe:
            divergent += 1
print(f"node_marks not in N(c): {divergent}")
assert divergent == 0, "audited nodes fall outside the enumerated N(c)"

node_marks not in N(c): 0


In [51]:
import plotly.express as px
import plotly.io as pio

px.bar(
    node_marks.filter(pl.col("decision") != "added")
    .with_columns(
        pl.col("type").str.replace_all("_", " ", literal=True).str.to_titlecase()
    )
    .group_by("type", "decision")
    .len()
    .sort(
        ["len", "type"],
        descending=[True, False],  # len absteigend, type aufsteigend
        maintain_order=True,  # stabil bei komplett gleichen Werten
    ),
    x="type",
    y="len",
    color_discrete_sequence=[
        pio.templates["thesis"].layout.colorway[3],
        pio.templates["thesis"].layout.colorway[4],
    ],
    color="decision",
).update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.0,
        xanchor="right",
        x=1.0,
    ),
    xaxis_title="",
    yaxis_title="Nodes",
    legend_title_text="",
    margin=dict(r=10, l=10, b=90, pad=10),
    height=400,
    width=800,
).update_xaxes(
    automargin=False
).show(
    config={"toImageButtonOptions": {"format": "svg"}}
)

In [52]:
with pl.Config(tbl_rows=100, fmt_str_lengths=400):
    print(
        annotation.select(
            "repository", "number", "categories", "attribution", "comment"
        )
    )

shape: (98, 5)
┌─────────────────────────┬────────┬────────────────────────┬─────────────┬────────────────────────┐
│ repository              ┆ number ┆ categories             ┆ attribution ┆ comment                │
│ ---                     ┆ ---    ┆ ---                    ┆ ---         ┆ ---                    │
│ str                     ┆ i64    ┆ list[str]              ┆ str         ┆ str                    │
╞═════════════════════════╪════════╪════════════════════════╪═════════════╪════════════════════════╡
│ microsoft/Olive         ┆ 2355   ┆ ["intrinsic"]          ┆ correct     ┆                        │
│ microsoft/Olive         ┆ 2438   ┆ ["intrinsic"]          ┆ correct     ┆                        │
│ microsoft/Olive         ┆ 1018   ┆ ["intrinsic", "other"] ┆ correct     ┆ The change also        │
│                         ┆        ┆                        ┆             ┆ contains work that     │
│                         ┆        ┆                        ┆             ┆ 

## Per-stratum error rates (posteriors)

Beta posteriors per stratum, without aggregation across strata or region weighting. Both require the stratum weights and release thresholds, which live downstream.

In [53]:
from scipy.stats import beta as beta_dist


class BetaPosterior:
    # Posterior mean and 95% credible interval of Beta(1 + a, 1 + b) per stratum,
    # with a as the rate's numerator and b its complement. Reported per stratum
    # without any region weighting or aggregation across strata.
    def rate(self, frame, a, b, name):
        alpha = 1 + frame[a].to_numpy()
        beta = 1 + frame[b].to_numpy()
        low, high = beta_dist.ppf([[0.025], [0.975]], alpha, beta)
        return frame.with_columns(
            (frame[a] / (frame[a] + frame[b])).round(3).alias(f"{name}_obs"),
            pl.Series(f"{name}_mean", beta_dist.mean(alpha, beta)).round(3),
            pl.Series(f"{name}_lo", low).round(3),
            pl.Series(f"{name}_hi", high).round(3),
        )


posterior = BetaPosterior()

# (a) Change-level error rate 1 - E[Y | rho_j] over all audited changes per stratum.
change_rate = posterior.rate(
    annotation.group_by("rho_bin")
    .agg(
        (pl.col("stage") != "passed").sum().alias("fails"),
        (pl.col("stage") == "passed").sum().alias("passed"),
    )
    .sort("rho_bin"),
    "fails",
    "passed",
    "error",
)

# (b) Node acceptance rate over the suggested nodes (Ỹ_c) per stratum.
node_rate = posterior.rate(
    node_marks.filter(pl.col("decision") != "added")
    .group_by("rho_bin")
    .agg(
        (pl.col("decision") == "accepted").sum().alias("accepted"),
        (pl.col("decision") == "rejected").sum().alias("rejected"),
    )
    .sort("rho_bin"),
    "accepted",
    "rejected",
    "accept",
)

with pl.Config(tbl_rows=-1):
    print("change-level error rate 1 - E[Y | rho_j]")
    print(change_rate)
    print("node acceptance rate")
    print(node_rate)

change-level error rate 1 - E[Y | rho_j]
shape: (3, 7)
┌─────────┬───────┬────────┬───────────┬────────────┬──────────┬──────────┐
│ rho_bin ┆ fails ┆ passed ┆ error_obs ┆ error_mean ┆ error_lo ┆ error_hi │
│ ---     ┆ ---   ┆ ---    ┆ ---       ┆ ---        ┆ ---      ┆ ---      │
│ i64     ┆ u32   ┆ u32    ┆ f64       ┆ f64        ┆ f64      ┆ f64      │
╞═════════╪═══════╪════════╪═══════════╪════════════╪══════════╪══════════╡
│ 0       ┆ 3     ┆ 14     ┆ 0.176     ┆ 0.211      ┆ 0.064    ┆ 0.414    │
│ 1       ┆ 9     ┆ 38     ┆ 0.191     ┆ 0.204      ┆ 0.105    ┆ 0.326    │
│ 2       ┆ 4     ┆ 30     ┆ 0.118     ┆ 0.139      ┆ 0.048    ┆ 0.267    │
└─────────┴───────┴────────┴───────────┴────────────┴──────────┴──────────┘
node acceptance rate
shape: (3, 7)
┌─────────┬──────────┬──────────┬────────────┬─────────────┬───────────┬───────────┐
│ rho_bin ┆ accepted ┆ rejected ┆ accept_obs ┆ accept_mean ┆ accept_lo ┆ accept_hi │
│ ---     ┆ ---      ┆ ---      ┆ ---        ┆ ---      

## Monotonicity of $\rho$

Whether $\rho$ carries information about label correctness is a premise of the stratification, not a result of it. The continuous $\rho$ is joined back from the confidence pool, since the annotation asset stores only the bin.

In [54]:
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

# rho is continuous but the annotation asset only stores rho_bin, so the continuous
# value is joined in from the confidence pool (rho = max per defect, as in the
# population). The target is the pass indicator: the change reached the node stage.
reliability = (
    pl.from_arrow(
        store.asset(
            key="confidence-change-reliability", version="5.0.0", partition="default"
        )
    )
    .filter(pl.col("rho").is_not_null())
    .sort("rho", descending=True)
    .unique(subset=["name", "defect"], keep="first")
    .select(
        pl.col("name").alias("repository"),
        pl.col("defect").alias("introducer"),
        "rho",
    )
)
scored = annotation.join(reliability, on=["repository", "introducer"], how="inner")


class Monotonicity:
    # Spearman rank correlation and AUROC of rho against a binary target, the AUROC
    # with a percentile bootstrap interval. Resamples missing either class are
    # skipped so roc_auc_score stays defined.
    def __init__(self, resamples=2000, seed=6031769):
        self._resamples = resamples
        self._rng = np.random.default_rng(seed)

    def report(self, rho, target, label):
        rho, target = np.asarray(rho, float), np.asarray(target, int)
        spearman, p = spearmanr(rho, target)
        auroc = roc_auc_score(target, rho)
        lo, hi = self._interval(rho, target)
        print(
            f"{label}: n={len(rho)}  spearman={spearman:.2f} (p={p:.2f})  "
            f"auroc={auroc:.2f} [{lo:.2f}, {hi:.2f}]"
        )

    def _interval(self, rho, target):
        draws = []
        index = np.arange(len(rho))
        for _ in range(self._resamples):
            pick = self._rng.choice(index, size=len(index), replace=True)
            if target[pick].min() == target[pick].max():
                continue
            draws.append(roc_auc_score(target[pick], rho[pick]))
        return np.percentile(draws, [2.5, 97.5])


tests = Monotonicity()

# Over all audited changes: does rho track whether the change passed?
passed = (scored["stage"] == "passed").cast(pl.Int64)
tests.report(scored["rho"], passed, "all (pass vs. fail)")

# Conditional slice: the changes with an intrinsic correction, target passed vs.
# stage_2 (a wrong introducer). Stage-1 failures are excluded.
conditional = scored.filter(pl.col("stage") != "stage_1")
tests.report(
    conditional["rho"],
    (conditional["stage"] == "passed").cast(pl.Int64),
    "intrinsic (passed vs. stage_2)",
)

all (pass vs. fail): n=98  spearman=0.12 (p=0.24)  auroc=0.59 [0.45, 0.74]
intrinsic (passed vs. stage_2): n=87  spearman=0.17 (p=0.13)  auroc=0.70 [0.49, 0.90]


## Unconditional $\hat \alpha$ counters

Estimated over all audited changes, not the passed ones alone: a change with $V_c \neq \mathit{none}$ has $Y_c = \emptyset$, so every node it released is incorrect. Failed changes carry no annotator decisions, so their $\tilde Y_c$ comes from the pipeline markings in the localization trace.

In [55]:
# Unconditional α̂ counters: the acceptance split is taken over all audited changes,
# not just the passed ones. A failed change has Y_c = ∅, so every released Ỹ_c node
# it carries counts incorrect. Its selections are empty (the audit stopped before the
# node stage), so its Ỹ_c is the pipeline marking from the localization trace — the
# same source the push reads to show the annotations.
sample = pl.from_arrow(
    store.asset(key="audit-change-sample", version="4.0.0", partition="default")
).select(
    pl.col("name").alias("repository"),
    "correction",
    pl.col("defect").alias("introducer"),
)
trace = pl.from_arrow(
    store.asset(
        key="localization-change-trace",
        version="3.1.0",
        partition="*",
        columns=["name", "correction", "defect", "locations"],
    )
).unique(subset=["name", "correction", "defect"])

failed = (
    annotation.filter(pl.col("stage") != "passed")
    .join(sample, on=["repository", "introducer"], how="left")
    .join(
        trace.select(
            pl.col("name").alias("repository"),
            "correction",
            pl.col("defect").alias("introducer"),
            "locations",
        ),
        on=["repository", "correction", "introducer"],
        how="left",
    )
)

fail_rows = []
for row in failed.iter_rows(named=True):
    for mark in json.loads(row["locations"] or "[]"):
        token = str(mark["line"])
        side = "base" if token[:1] == "D" else "after"
        number = int(token[1:] if token[:1] in ("L", "D") else token)
        node = projection(row["repository"]).project(
            row["introducer"], mark["file"], number, side
        )
        if node is not None:
            fail_rows.append(
                {
                    "repository": row["repository"],
                    "rho_bin": row["rho_bin"],
                    "number": row["number"],
                    "node": f"{node.path}:{node.start}:{node.end}:{node.type}",
                }
            )

# Deduplicate to nodes per failed change; every fail node is incorrect.
fail_nodes = (
    pl.DataFrame(fail_rows)
    .unique(subset=["repository", "number", "node"])
    .group_by("rho_bin")
    .len()
    .rename({"len": "fail"})
)

# a_j = rejected suggested nodes of the passed changes + all fail nodes; b_j =
# accepted suggested nodes. Added nodes stay out, α̂ conditions only on Ỹ_c.
passed_counts = (
    node_marks.filter(pl.col("decision") != "added")
    .group_by("rho_bin")
    .agg(
        (pl.col("decision") == "rejected").sum().alias("rejected"),
        (pl.col("decision") == "accepted").sum().alias("accepted"),
    )
)
counts = (
    passed_counts.join(fail_nodes, on="rho_bin", how="left")
    .fill_null(0)
    .with_columns(
        (pl.col("rejected") + pl.col("fail")).alias("a_j"),
        pl.col("accepted").alias("b_j"),
    )
    .with_columns(
        (pl.col("a_j") + pl.col("b_j")).alias("marked_total"),
        (pl.col("a_j") / (pl.col("a_j") + pl.col("b_j"))).round(2).alias("fdr_uncond"),
    )
    .sort("rho_bin")
)

posterior = BetaPosterior()
table = posterior.rate(
    counts.select("rho_bin", "marked_total", "a_j", "b_j", "fdr_uncond"),
    "a_j",
    "b_j",
    "alpha",
)
with pl.Config(tbl_rows=-1):
    print(table)

a_total, b_total = counts["a_j"].sum(), counts["b_j"].sum()
print(
    f"unweighted overall  a={a_total}  b={b_total}  "
    f"fdr={a_total / (a_total + b_total):.2f}"
)

shape: (3, 9)
┌─────────┬──────────────┬─────┬─────┬───┬───────────┬────────────┬──────────┬──────────┐
│ rho_bin ┆ marked_total ┆ a_j ┆ b_j ┆ … ┆ alpha_obs ┆ alpha_mean ┆ alpha_lo ┆ alpha_hi │
│ ---     ┆ ---          ┆ --- ┆ --- ┆   ┆ ---       ┆ ---        ┆ ---      ┆ ---      │
│ i64     ┆ u32          ┆ u32 ┆ u32 ┆   ┆ f64       ┆ f64        ┆ f64      ┆ f64      │
╞═════════╪══════════════╪═════╪═════╪═══╪═══════════╪════════════╪══════════╪══════════╡
│ 0       ┆ 24           ┆ 10  ┆ 14  ┆ … ┆ 0.417     ┆ 0.423      ┆ 0.244    ┆ 0.613    │
│ 1       ┆ 119          ┆ 60  ┆ 59  ┆ … ┆ 0.504     ┆ 0.504      ┆ 0.416    ┆ 0.593    │
│ 2       ┆ 79           ┆ 41  ┆ 38  ┆ … ┆ 0.519     ┆ 0.519      ┆ 0.41     ┆ 0.626    │
└─────────┴──────────────┴─────┴─────┴───┴───────────┴────────────┴──────────┴──────────┘
unweighted overall  a=111  b=111  fdr=0.50


## Unconditional $\hat \alpha$ at module level

The same estimand aggregated to the affected file. A module counts correct once any of its markings was accepted, so the aggregation is optimistic and bounds the node-level rate from below.

In [56]:
# The same unconditional α̂, but aggregated to the module: a node is the affected
# file (change, path) rather than a syntax node. Pure aggregation of the existing
# markings, no lift. A module counts correct once any of its suggested markings was
# accepted, and incorrect only when all were rejected -- accepted wins.
passed_modules = (
    marks.filter(pl.col("origin") == "suggested")
    .group_by("repository", "number", "rho_bin", "path")
    .agg((pl.col("status") == "accepted").any().alias("any_accepted"))
    .with_columns(
        pl.when(pl.col("any_accepted"))
        .then(pl.lit("accepted"))
        .otherwise(pl.lit("rejected"))
        .alias("decision")
    )
)

# Failed changes have Y_c = ∅, so every module they marked is incorrect.
fail_modules = (
    failed.select("repository", "rho_bin", "number", "locations")
    .with_columns(
        pl.col("locations").map_elements(
            lambda value: [m["file"] for m in json.loads(value or "[]")],
            return_dtype=pl.List(pl.String),
        )
    )
    .explode("locations")
    .drop_nulls("locations")
    .unique(subset=["repository", "number", "locations"])
    .group_by("rho_bin")
    .len()
    .rename({"len": "fail"})
)

module_counts = (
    passed_modules.group_by("rho_bin")
    .agg(
        (pl.col("decision") == "rejected").sum().alias("rejected"),
        (pl.col("decision") == "accepted").sum().alias("accepted"),
    )
    .join(fail_modules, on="rho_bin", how="left")
    .fill_null(0)
    .with_columns(
        (pl.col("rejected") + pl.col("fail")).alias("a_j"),
        pl.col("accepted").alias("b_j"),
    )
    .with_columns(
        (pl.col("a_j") + pl.col("b_j")).alias("marked_total"),
        (pl.col("a_j") / (pl.col("a_j") + pl.col("b_j"))).round(2).alias("fdr_uncond"),
    )
    .sort("rho_bin")
)

module_table = posterior.rate(
    module_counts.select("rho_bin", "marked_total", "a_j", "b_j", "fdr_uncond"),
    "a_j",
    "b_j",
    "alpha",
)
with pl.Config(tbl_rows=-1):
    print(module_table)

a_total, b_total = module_counts["a_j"].sum(), module_counts["b_j"].sum()
print(
    f"unweighted overall  a={a_total}  b={b_total}  "
    f"fdr={a_total / (a_total + b_total):.2f}"
)

shape: (3, 9)
┌─────────┬──────────────┬─────┬─────┬───┬───────────┬────────────┬──────────┬──────────┐
│ rho_bin ┆ marked_total ┆ a_j ┆ b_j ┆ … ┆ alpha_obs ┆ alpha_mean ┆ alpha_lo ┆ alpha_hi │
│ ---     ┆ ---          ┆ --- ┆ --- ┆   ┆ ---       ┆ ---        ┆ ---      ┆ ---      │
│ i64     ┆ u32          ┆ u32 ┆ u32 ┆   ┆ f64       ┆ f64        ┆ f64      ┆ f64      │
╞═════════╪══════════════╪═════╪═════╪═══╪═══════════╪════════════╪══════════╪══════════╡
│ 0       ┆ 18           ┆ 6   ┆ 12  ┆ … ┆ 0.333     ┆ 0.35       ┆ 0.163    ┆ 0.566    │
│ 1       ┆ 55           ┆ 16  ┆ 39  ┆ … ┆ 0.291     ┆ 0.298      ┆ 0.188    ┆ 0.422    │
│ 2       ┆ 44           ┆ 12  ┆ 32  ┆ … ┆ 0.273     ┆ 0.283      ┆ 0.164    ┆ 0.419    │
└─────────┴──────────────┴─────┴─────┴───┴───────────┴────────────┴──────────┴──────────┘
unweighted overall  a=34  b=83  fdr=0.29


## Omission $\gamma^{+}$ counters

The omission rate on the positive region: among the unmarked nodes of a released-positive
change, how many the annotator added ($A_c = Y_c \setminus \tilde Y_c$) against how many
were rightly left unmarked. Its counterpart $\gamma^{-}$ lives on the negative region
($\tilde Y_c = \emptyset$); both are noise-model parameters. It rests on the marking being
complete with respect to the stated root cause, so an unmarked node counts as free of it,
which is what enters $b_j$.

In [57]:
# a_j = added nodes A_c; b_j = the rightly-unmarked nodes |N(c)| - |Ỹ_c| - |A_c|.
per_change = (
    node_marks.group_by("repository", "number", "rho_bin")
    .agg(
        (pl.col("decision") != "added").sum().alias("marked"),
        (pl.col("decision") == "added").sum().alias("added"),
    )
    .join(n_frame, on=["repository", "number"], how="left")
)

# Conditional (passed only): diagnostic.
conditional = (
    per_change.join(
        annotation.select("repository", "number", "stage"), on=["repository", "number"]
    )
    .filter(pl.col("stage") == "passed")
    .group_by("rho_bin")
    .agg(
        pl.col("added").sum().alias("a_j"),
        (pl.col("n_c") - pl.col("marked") - pl.col("added")).sum().alias("b_j"),
    )
    .sort("rho_bin")
)

# Unconditional (result): all released positives including failed changes. A failed
# change has A_c = 0 and its Ỹ_c is the pipeline marking from the trace, so it adds
# n_c - |Ỹ_c_trace| rightly-unmarked nodes to b_j.
fail_marked = (
    pl.DataFrame(fail_rows)
    .unique(subset=["repository", "number", "node"])
    .group_by("repository", "number", "rho_bin")
    .len()
    .rename({"len": "marked"})
)
fail_frame = (
    failed.select("repository", "number", "rho_bin")
    .join(
        fail_marked.select("repository", "number", "marked"),
        on=["repository", "number"],
        how="left",
    )
    .join(n_frame, on=["repository", "number"], how="left")
    .fill_null(0)
    .with_columns(pl.lit(0).alias("added"))
    .group_by("rho_bin")
    .agg(
        pl.col("added").sum().alias("a_j"),
        (pl.col("n_c") - pl.col("marked")).sum().alias("b_j"),
    )
)
unconditional = (
    conditional.join(fail_frame, on="rho_bin", how="left", suffix="_fail")
    .fill_null(0)
    .with_columns(
        (pl.col("a_j") + pl.col("a_j_fail")).alias("a_j"),
        (pl.col("b_j") + pl.col("b_j_fail")).alias("b_j"),
    )
    .select("rho_bin", "a_j", "b_j")
    .sort("rho_bin")
)

gamma = BetaPosterior()
print("conditional (passed only, diagnostic):")
print(gamma.rate(conditional, "a_j", "b_j", "gamma_plus"))
print("unconditional (all released positives incl. fails):")
table = gamma.rate(unconditional, "a_j", "b_j", "gamma_plus")
print(table)
a_total, b_total = unconditional["a_j"].sum(), unconditional["b_j"].sum()
print(
    f"unweighted overall  a={a_total}  b={b_total}  gamma+={a_total / (a_total + b_total):.4f}"
)

conditional (passed only, diagnostic):
shape: (3, 7)
┌─────────┬─────┬───────┬────────────────┬─────────────────┬───────────────┬───────────────┐
│ rho_bin ┆ a_j ┆ b_j   ┆ gamma_plus_obs ┆ gamma_plus_mean ┆ gamma_plus_lo ┆ gamma_plus_hi │
│ ---     ┆ --- ┆ ---   ┆ ---            ┆ ---             ┆ ---           ┆ ---           │
│ i64     ┆ u32 ┆ i64   ┆ f64            ┆ f64             ┆ f64           ┆ f64           │
╞═════════╪═════╪═══════╪════════════════╪═════════════════╪═══════════════╪═══════════════╡
│ 0       ┆ 2   ┆ 3502  ┆ 0.001          ┆ 0.001           ┆ 0.0           ┆ 0.002         │
│ 1       ┆ 2   ┆ 10459 ┆ 0.0            ┆ 0.0             ┆ 0.0           ┆ 0.001         │
│ 2       ┆ 3   ┆ 5459  ┆ 0.001          ┆ 0.001           ┆ 0.0           ┆ 0.002         │
└─────────┴─────┴───────┴────────────────┴─────────────────┴───────────────┴───────────────┘
unconditional (all released positives incl. fails):
shape: (3, 7)
┌─────────┬─────┬───────┬────────────────┬──